#Основы работы с Apache Spark


In [ ]:
# 1. Установка Java и Spark
#ДАННАЯ ЯЧЕЙКА ПЛОХО РАБОТАЕТ. УСТАНОВКА ПЕРЕНЕСЕНА НА СЛЕДУЮЩИЕ ЯЧЕЙКИ.
# Устанавливаем OpenJDK 21
!apt-get install openjdk-21-jdk-headless -qq > /dev/null

# Устанавливаем Spark 4.0.1
!wget -q https://archive.apache.org/dist/spark/spark-4.0.1/spark-4.0.1-bin-hadoop3.tgz -O spark.tgz
!tar xf spark.tgz

# Устанавливаем findspark
!pip install -q findspark

# 2. Настройка окружения

import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-4.0.1-bin-hadoop3"

# 3. Инициализация Spark

import findspark
findspark.init()

from pyspark.sql import SparkSession

# Создаём SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("DataEngineeringCourse") \
    .getOrCreate()

# получаем SparkContext для RDD-заданий (5.1)
sc = spark.sparkContext

# 4. Проверка, что Spark работает

print("=" * 60)
print("Проверка Spark")
print("=" * 60)
print(f"Spark version: {spark.version}")
print(f"SparkContext: {sc}")
print(f"Master: {sc.master}")
print(f"App Name: {sc.appName}")
print(f"Default Parallelism: {sc.defaultParallelism}")

In [81]:
# Устанавливаем OpenJDK 21
!apt-get install -y openjdk-21-jdk-headless -qq > /dev/null

# PySpark через pip - без скачивания полного архива Spark
!pip install -q pyspark==4.0.1

In [82]:
#Проверка, что установилось
!java -version

[0.003s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.003s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [83]:
import os

# Настройка окружения
os.environ.pop("SPARK_HOME", None)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

print("SPARK_HOME =", os.environ.get("SPARK_HOME"))
print("JAVA_HOME  =", os.environ.get("JAVA_HOME"))

SPARK_HOME = None
JAVA_HOME  = /usr/lib/jvm/java-21-openjdk-amd64


In [84]:
from pyspark.sql import SparkSession
# Создаём SparkSession, получаем SparkContext для RDD-заданий (5.1), Проверка, что Spark работает
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("DataEngineeringCourse") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

sc = spark.sparkContext

print("=" * 60)
print("Проверка Spark")
print("=" * 60)
print(f"Spark version:        {spark.version}")
print(f"SparkContext:         {sc}")
print(f"Master:               {sc.master}")
print(f"App Name:             {sc.appName}")
print(f"Default Parallelism:  {sc.defaultParallelism}")

Проверка Spark
Spark version:        4.0.1
SparkContext:         <SparkContext master=local[*] appName=DataEngineeringCourse>
Master:               local[*]
App Name:             DataEngineeringCourse
Default Parallelism:  2


In [85]:
#Загрузка датасета
from pyspark.sql.functions import col, avg, count, desc, round

games = spark.read.csv("/content/drive/MyDrive/Универ/Инжиниринг данных/variant_8_games.csv", header=True, inferSchema=True, quote='"', escape='"')
games.printSchema()
games.show(5, truncate=False)

root
 |-- app_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- price: double (nullable = true)
 |-- positive_ratio: integer (nullable = true)
 |-- release_date: date (nullable = true)

+------+------+----------+----------+-----+--------------+------------+
|app_id|name  |developer |genre     |price|positive_ratio|release_date|
+------+------+----------+----------+-----+--------------+------------+
|1     |Game_0|Ubisoft   |Simulation|30.14|94            |2017-05-15  |
|2     |Game_1|CD Projekt|Shooter   |0.05 |40            |2019-08-11  |
|3     |Game_2|CD Projekt|Strategy  |44.56|22            |2018-05-09  |
|4     |Game_3|CD Projekt|Strategy  |20.06|73            |2018-04-24  |
|5     |Game_4|Ubisoft   |Adventure |38.6 |97            |2023-08-27  |
+------+------+----------+----------+-----+--------------+------------+
only showing top 5 rows


In [88]:
#Rdd из row-объектов + вспомогательная функция
games_rdd = games.rdd
def rdd_show(rdd):
    for i in rdd.collect(): print(i)

In [89]:
#Задание 1: кол-во игр по разработчикам

dev_counts = (
    games_rdd
    .filter(lambda x: x['developer'] is not None)
    .map(lambda x: (x["developer"], 1))
    .reduceByKey(lambda x, y: x+y)
    .sortBy(lambda x: x[1], ascending=False)
)
rdd_show(dev_counts)

('Ubisoft', 186)
('EA', 181)
('Valve', 178)
('CD Projekt', 165)
('Square Enix', 150)
('Rockstar', 140)


In [90]:
#Задание 2: средняя цена и количество игр по жанрам
genre_count = (
    games_rdd
    .filter(lambda x: x["genre"] is not None)
    .map(lambda x: (x['genre'], 1))
    .reduceByKey(lambda x, y: x+y)
    )

genre_price_sum = (
    games_rdd
    .filter(lambda x: x['genre'] is not None and x['price'] is not None)
    .map(lambda r: (r['genre'], (float(r['price']), 1)))
    .reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1]))
    .map(lambda x: (x[0], x[1][0] / x[1][1] if x[1][1] > 0 else 0.0))
)

genre_stats = (
    genre_count
    .join(genre_price_sum)
    .map(lambda x: (x[0], x[1][1], x[1][0]))
    .sortBy(lambda x: x[2], ascending=False)
)

for genre, avg_price, cnt in genre_stats.collect():
  print(f"{genre}: сред. цена = {avg_price:.4f}, кол-во игр = {cnt}")

Shooter: сред. цена = 31.0038, кол-во игр = 182
Adventure: сред. цена = 30.1757, кол-во игр = 171
Action: сред. цена = 27.8308, кол-во игр = 167
Simulation: сред. цена = 29.3354, кол-во игр = 162
Strategy: сред. цена = 27.5059, кол-во игр = 159
RPG: сред. цена = 33.2657, кол-во игр = 159


In [91]:
# Альтернативное решение задания 2 через DataFrame API
genre_stats_df = (
    games
    .filter(col("genre").isNotNull() & col("price").isNotNull())
    .groupBy("genre")
    .agg(
        round(avg("price"), 4).alias("avg_price"),
        count("*").alias("games_count")
    )
    .orderBy(desc("games_count"))
)
genre_stats_df.show(truncate=False)

+----------+---------+-----------+
|genre     |avg_price|games_count|
+----------+---------+-----------+
|Shooter   |31.0038  |182        |
|Adventure |30.1757  |171        |
|Action    |27.8308  |167        |
|Simulation|29.3354  |162        |
|Strategy  |27.5059  |159        |
|RPG       |33.2657  |159        |
+----------+---------+-----------+



In [92]:
#Задание 3: топ 5 игр с наиб. positive_ratio
top5 = (
    games_rdd
    .filter(lambda r: r["positive_ratio"] is not None and r["name"] is not None)
    .map(lambda r: (float(r["positive_ratio"]), r["name"]))
    .sortByKey(ascending=False)
    .take(5)
)
for r in top5: print(r)
spark.stop()

(99.0, 'Game_84')
(99.0, 'Game_191')
(99.0, 'Game_373')
(99.0, 'Game_490')
(99.0, 'Game_496')
